<a href="https://colab.research.google.com/github/Illusionsp/gdsc_study_session_ML_g1/blob/main/Fake_News_Detection_TensorFlow_Dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import re  # <-- We added this to use RegEx!
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

print("Loading datasets from Google Drive...")

# Now that the drive is mounted, this will work!
fake = pd.read_csv("/content/drive/MyDrive/Fake.csv")
real = pd.read_csv("/content/drive/MyDrive/True.csv")

# Add Labels: 1 = Fake News, 0 = Real News
fake["label"] = 1
real["label"] = 0

# Combine into one big dataframe
df = pd.concat([fake, real])[["text", "label"]]

# --- NEW PROFESSIONAL DATA CLEANING STEP ---
print("🧹 Cleaning Dataset Bias (Removing Publisher Tags)...")
df["text"] = df["text"].str.replace(r'^.*?\(Reuters\)\s*-\s*', '', regex=True)
# -------------------------------------------

print("Converting text to numbers using TF-IDF...")
vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")
X = vectorizer.fit_transform(df["text"]).toarray()
y = df["label"].values

# Split into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data ready! Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

In [ ]:
# --- CELL 3: Build the TensorFlow Model ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("Building the custom TensorFlow model with Dropout layers...")

model = Sequential([
    # Input layer (3000 features from TF-IDF) connected to 128 neurons
    Dense(128, activation='relu', input_shape=(3000,)),
    # DROPOUT: Randomly turns off 50% of neurons to prevent memorization
    Dropout(0.5),

    # Hidden layer with 64 neurons
    Dense(64, activation='relu'),
    # DROPOUT: Randomly turns off 30% of neurons
    Dropout(0.3),

    # Output layer: 1 neuron predicting between 0 (Real) and 1 (Fake)
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# --- CELL 4: Train the Model ---
import matplotlib.pyplot as plt

print("Training started! Watch the accuracy go up...")

# Train the model (using 80% of our training data to train, and 20% to validate)
history = model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)

# Plot the training results to show your professor
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy Over Time')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
# --- CELL 5: Evaluate and Interactive Tester ---
from sklearn.metrics import confusion_matrix
import ipywidgets as widgets
from IPython.display import display

# 1. Final Grade on Test Data
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n=============================")
print(f"🏆 FINAL TEST ACCURACY: {accuracy * 100:.2f}%")
print(f"=============================\n")

# 2. Interactive Tester Widget
def predict_news(text):
    vec = vectorizer.transform([text]).toarray()
    prediction = model.predict(vec, verbose=0)[0][0]
    return "🚨 FAKE NEWS" if prediction > 0.5 else "✅ REAL NEWS"

print("--- TEST IT YOURSELF ---")
headline_input = widgets.Text(
    placeholder='Type a news headline here...',
    description='Headline:',
    layout=widgets.Layout(width='80%')
)
predict_button = widgets.Button(description="Check News", button_style='info')
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        if not headline_input.value.strip():
            print("Please type a headline first!")
            return
        result = predict_news(headline_input.value)
        print(f"Prediction: {result}")

predict_button.on_click(on_click)
display(headline_input, predict_button, output)

# 3. Save your masterpiece (UPDATED TO .keras FORMAT)
model.save("tf_custom_fake_news.keras")
print("\nModel saved as 'tf_custom_fake_news.keras'. You are officially done!")